# Pandas Coding Challenges

This is a workout, not a lecture. Ten short, self-contained problems exercise the Pandas skills that interviewers and everyday analysis depend on — reading data, ranking, grouping, filling, merging, pivoting, rolling windows, ranking within groups, reshaping, and memory efficiency. Every solution cell ends with an `assert` statement, so the notebook checks *you*: if the assertion passes silently, your answer is correct; if it raises, your logic is off. That self-verifying style is deliberate — it mirrors unit-tested production code and trains the habit of reasoning about the exact value a function should return before trusting it.

The problems are deliberately tiny so that the *concept* is the star. Challenge 1 inspects a frame's `shape`, `dtypes`, and null counts — the three diagnostics you run on every new dataset. Challenge 2 uses `nlargest` to pull the top rows by a column. Challenge 3 pairs a groupby with named aggregations (`size` and `sum` at once). Challenge 4 fills missing values *per group* with `transform` — a deceptively powerful pattern. Challenge 5 merges two frames with an `indicator` column and counts unmatched rows, the standard way to audit a join. Challenge 6 builds a pivot table with `margins`, and Challenge 7 computes a 3-period rolling mean, the building block of smoothing and moving averages. Challenge 8 ranks rows *within each group*, Challenge 9 melts a wide frame to long/tidy form, and Challenge 10 downcasts a numeric column and confirms the memory saving.

Each challenge is presented as: the brief, a short explanation of the concept and the relevant API, and a hint about which call to reach for. The solution code follows immediately, ending in its assertion. Attempt the problem before reading ahead where you can — retrieval, not re-reading, is what makes these patterns stick. At a realistic screening pace, aim for about ten minutes per challenge; note any that take longer and revisit them at the end.

## Setup

Start by importing the two libraries you will use throughout — `numpy` for array work and `pandas` for the frames — and configuring the display. `pd.set_option('display.width', 120)` widens the console output so wide frames print on one screen instead of wrapping, and `pd.set_option('display.max_columns', 30)` raises the column cap so challenges with more columns show them all. These options only affect *presentation*; they never change the data.

The line `np.random.seed(0)` fixes NumPy's random generator. None of these ten challenges depend on randomness, but seeding is a habit worth keeping: any time you shuffle, sample, or split data, the same seed reproduces the exact same result, which makes debugging and grading deterministic. Finally, `print('pandas', pd.__version__)` records the Pandas version — behaviour genuinely differs between releases (for example, the `errors`/`regex` defaults of some string and datetime functions), so knowing the version heads off mysterious mismatches.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
np.random.seed(0)
print('pandas', pd.__version__)

## Challenge 1 — Inspect a frame

**Brief:** Read a CSV and report its shape, dtypes, and null counts.

Any first look at unfamiliar tabular data follows the same three-question ritual: *How big is it? What type is each column? Where are the holes?* The API answers are `df.shape` (a `(rows, columns)` tuple), `df.dtypes` (the dtype per column — `int64`, `float64`, `object`, `datetime64[ns]`, ...), and `df.isna().sum()` (the count of missing entries per column; `.isna()`/`.isnull()` are aliases for the same mask).

With real data you would begin with `pd.read_csv('file.csv')`, but here the frame is built inline as `{'a': [1, None], 'b': ['x', 'y']}`. Notice that a Python `None` inside an integer column forces the whole column to `float64` (or `object`), because NumPy has no integer representation for 'missing' — a subtlety that explains why nulls silently promote dtypes. `df.dtypes.to_dict()` and `df.isna().sum().to_dict()` convert the Series results to plain dictionaries, which is convenient for programmatic checks. The assertion confirms the frame is 2x2, teaching you to verify structure before analysing content.

In [ ]:
df = pd.DataFrame({'a': [1, None], 'b': ['x', 'y']})
assert df.shape == (2, 2)
print(df.dtypes.to_dict(), df.isna().sum().to_dict())

## Challenge 2 — Top rows by a column

**Brief:** Find the top three rows by a column.

The idiomatic call is `df.nlargest(3, 'v')` — it returns the three rows with the largest `v`, already sorted descending. It is essentially a specialised sort with a limit, and it is usually faster than `df.sort_values('v', ascending=False).head(3)` because `nlargest` can select the top-k without fully ordering the rest. It handles ties gracefully and can break ties with a second column (`nlargest(3, ['v','w'])`).

Analogues worth knowing: `nsmallest` for the bottom-k, and `sort_values` when you want the *entire* ordering. The example frame `{'v': [5, 9, 1, 7]}` makes the result easy to eyeball — the top three are `9, 7, 5`. The assertion checks `top.iloc[0]['v'] == 9`, mixing positional selection (`iloc[0]` — the first row after `nlargest`) with label selection (`['v']`). That pairing is the key insight: `nlargest` reorders rows, so the returned index follows the new order (unless you reset it), which is exactly why the assertion works.

In [ ]:
df = pd.DataFrame({'v': [5, 9, 1, 7]})
top = df.nlargest(3, 'v')
assert top.iloc[0]['v'] == 9
print(top)

## Challenge 3 — Two aggregations at once

**Brief:** Group by one column and compute two aggregations.

Grouping splits a frame by a key, applies a function to each group, and recombines — the split-apply-combine pattern. The special form used here is **named aggregation**: `df.groupby('g').agg(n=('v', 'size'), total=('v', 'sum'))` produces a frame whose columns are named `n` and `total`, with `n` computed by applying `'size'` to `v` and `total` by applying `'sum'`. Naming your outputs at aggregation time is far cleaner than post-renaming an unnamed result, and it composes for as many aggregations as you like.

Why `size` rather than `count` for `n`? Because `size` counts rows in each group including nulls, while `count` counts non-null values — a meaningful distinction on real, holey data. For `{'g': ['a','a','b'], 'v': [1,3,5]}` the groups are `a → [1,3]` and `b → [5]`, so `total` is 4 and 5 respectively. The assertion pins `out.loc['a', 'total'] == 4`, and because `g` became the index, `.loc` reads the group key directly. Related helpers: `.mean()`, `.max()`, and passing a dict or list of functions instead of the named form.

In [ ]:
df = pd.DataFrame({'g': ['a', 'a', 'b'], 'v': [1, 3, 5]})
out = df.groupby('g').agg(n=('v', 'size'), total=('v', 'sum'))
assert out.loc['a', 'total'] == 4
print(out)

## Challenge 4 — Fill nulls with the group mean

**Brief:** Fill missing values per group with the group mean.

The pattern is `df.groupby('g')['v'].transform(lambda s: s.fillna(s.mean()))`. Two pieces make it work. First, **`transform`** returns a result the *same shape* as the input, aligned back to the original rows — unlike `agg`, which collapses each group to one value. Second, the lambda receives each group's Series, fills its nulls with that group's own mean, and returns the filled group; `transform` stitches the pieces back in place. The result: `v` is repaired using *its own group's* average, not a global average.

The example `{'g': ['a','a','b'], 'v': [1.0, None, 4.0]}` has group `a` with mean 1.0, so the missing middle value becomes 1.0 — exactly what the assertion checks. This same transform-based idiom generalises to standardisation (`(s - s.mean())/s.std()`), z-scores per group, and any group-relative scaling, because `transform` guarantees the output slots back onto the original index. Watch one detail: after `fillna`, the column dtype may shift from float to int in some cases, so confirm with `df.dtypes` when downstream code is type-sensitive.

In [ ]:
df = pd.DataFrame({'g': ['a', 'a', 'b'], 'v': [1.0, None, 4.0]})
df['v'] = df.groupby('g')['v'].transform(lambda s: s.fillna(s.mean()))
assert df.loc[1, 'v'] == 1.0
print(df)

## Challenge 5 — Merge and audit unmatched rows

**Brief:** Merge two frames and count unmatched rows.

`a.merge(b, on='k', how='left', indicator=True)` performs a **left join** on the shared key `k`: every row of `a` is kept, and matching columns from `b` are attached where `k` agrees. The star feature is `indicator=True`, which adds a `_merge` column with three possible values — `'both'`, `'left_only'`, `'right_only'` — telling you the provenance of every output row. That column is the fastest way to *audit* a join: far too many analyses break because a merge unexpectedly duplicated rows or silently dropped records, and `_merge` surfaces exactly that.

Here `a` has keys `[1,2,3]` and `b` has just `[2]`, so with `how='left'` the output has three rows: keys 1 and 3 are `left_only` (no match) and key 2 is `both` (bringing `v = 9`). Counting unmatched rows is therefore `(m['_merge'] == 'left_only').sum() == 2`, and the assertion confirms it. Remember the join families and their row-keeping rules — `inner` keeps only matches, `left` keeps all of the left frame, `outer` keeps everything — and note that merges on non-unique keys can multiply rows, which `_merge` combined with a shape check will reveal.

In [ ]:
a = pd.DataFrame({'k': [1, 2, 3]})
b = pd.DataFrame({'k': [2], 'v': [9]})
m = a.merge(b, on='k', how='left', indicator=True)
assert (m['_merge'] == 'left_only').sum() == 2
print(m)

## Challenge 6 — Pivot table with margins

**Brief:** Create a pivot table with margins.

`df.pivot_table(index='r', columns='c', values='v', aggfunc='sum', margins=True)` reshapes the frame into a two-dimensional summary: one row per `r` value, one column per `c` value, and each cell the summed `v` for that combination. This is groupby + unstack in a single call, and it is the workhorse behind cross-tab reports. The expression's three key arguments are `index` (row keys), `columns` (column keys), and `values` (what gets aggregated); `aggfunc` chooses the reduction, `'sum'` here, with `'mean'` as the default when omitted.

**`margins=True`** adds grand-total rows and columns labelled `'All'` — the bottom-right cell aggregates everything, which the assertion pins to `3` (`1 + 2`). Margins are exactly the totals a manager expects at the edges of a spreadsheet pivot, and Pandas computes them consistently with the cell aggregation. Two practical knobs to remember: `fill_value=0` replaces empty combinations with zero so the grid has no gaps, and `margins_name=` renames 'All' if you need a different label. This challenge is a compact rehearsal of the pivot-table session, and `pd.crosstab` is the close cousin when you only need counts.

In [ ]:
df = pd.DataFrame({'r': ['x', 'y'], 'c': ['p', 'p'], 'v': [1, 2]})
pt = df.pivot_table(index='r', columns='c', values='v',
                    aggfunc='sum', margins=True)
assert pt.loc['All', 'All'] == 3
print(pt)

## Challenge 7 — Rolling mean

**Brief:** Compute a 3-period rolling mean.

`s.rolling(3).mean()` creates a sliding window of width three and averages the values inside it at every position. On `[1, 2, 3, 4, 5]` the outputs are: position 0 → `NaN` (not enough history), position 1 → `NaN`, position 2 → mean of `[1,2,3] = 2.0`, position 3 → mean of `[2,3,4] = 3.0`, position 4 → mean of `[3,4,5] = 4.0`. The first two are `NaN` because a full window cannot form yet — the default requires `window` non-null observations. Hence the assertion checks `r.iloc[2] == 2.0`, the first *valid* value.

`rolling` is the foundation of smoothing and moving averages used everywhere in time series and signal processing: financial indicators, sensor de-noising, and trend extraction. Useful options: `min_periods=1` makes the window emit a value sooner (averaging whatever is available); `center=True` aligns the window's centre with the label rather than its right edge; and `rolling(...).std()`/`.sum()` compute other rolling statistics with the identical API. Remember to sort chronologically before rolling if the data is a time series — windowing unordered data produces meaningless numbers.

In [ ]:
s = pd.Series([1, 2, 3, 4, 5], dtype=float)
r = s.rolling(3).mean()
assert r.iloc[2] == 2.0
print(r.tolist())

## Challenge 8 — Rank within groups

**Brief:** Rank rows within each group.

The expression `df.groupby('g')['v'].rank(ascending=False)` computes a rank *inside each group* and returns it aligned to the original rows. With `ascending=False`, the largest value in a group gets rank `1`. The example `{'g': ['a','a','b'], 'v': [1, 9, 4]}` has group `a` containing `[1, 9]`, so `9` ranks 1 and `1` ranks 2; group `b` has a single `4` ranking 1. Because `groupby(...).rank()` returns a Series of the same length, assigning it to `df['r']` drops the ranks neatly beside the originals — the assertion checks `df.loc[1, 'r'] == 1.0` (row 1 is the `9` in group `a`).

The default `method='average'` gives tied values the average of their ranks; alternatives matter. `method='min'` assigns all ties the lowest rank (like Olympic standings), `'dense'` assigns consecutive ranks with no gaps, and `'first'` breaks ties by row order. Group-wise ranking powers percentiles-within-cohort, leaderboards, top-N-per-group selection (`df[df.groupby('g')['v'].rank(...) <= 3]`), and competition scoring. It is one of the clearest examples of how `groupby` + `transform`-style functions keep every row while adding group-relative context.

In [ ]:
df = pd.DataFrame({'g': ['a', 'a', 'b'], 'v': [1, 9, 4]})
df['r'] = df.groupby('g')['v'].rank(ascending=False)
assert df.loc[1, 'r'] == 1.0
print(df)

## Challenge 9 — Wide to long

**Brief:** Convert a wide frame to long format.

`wide.melt(id_vars='id')` moves the frame from **wide** (one row per entity, one column per attribute) into **long or tidy** form (one row per attribute-value observation). `id_vars='id'` names the column(s) that stay fixed as identifiers; every *other* column becomes a pair of values under the two produced columns `variable` (the former column name) and `value` (its cell). The input `{'id': [1], 'a': [10], 'b': [20]}` therefore melts into two rows — `(id=1, variable='a', value=10)` and `(id=1, variable='b', value=20)` — which the assertion confirms via `len(long) == 2`.

Why reshape at all? Machines and libraries — plotting libraries, groupby, statistical models, and most Python APIs — prefer long data; humans prefer wide data. `melt` is the standard bridge, with `pd.melt` applied to a wide frame the usual entry point. There is a **tidiness rule** behind the vocabulary: each variable its own column, each observation its own row, each type of observational unit its own table. `var_name=` and `value_name=` rename the generated columns, and specifying `value_vars=` melts only a chosen subset. Its inverse is `pivot`/`pivot_table`, which widens long data back — the two calls are the doorways between the two shapes.

In [ ]:
wide = pd.DataFrame({'id': [1], 'a': [10], 'b': [20]})
long = wide.melt(id_vars='id')
assert len(long) == 2
print(long)

## Challenge 10 — Downcast and save memory

**Brief:** Downcast a numeric column and confirm memory shrinks.

The frame starts with a plain Python `range`, which Pandas materialises as the wide default `int64`. `pd.to_numeric(df['v'], downcast='unsigned')` re-types that column to the *smallest unsigned integer* that can hold its values — for `0..999` that is `uint16`, halving the per-value cost versus `int64`. `df.memory_usage(deep=True).sum()` measures the total bytes before and after; `deep=True` matters, because it counts the memory of object contents too, not just pointer arrays. The assertion simply confirms `after < before` — the downcast paid off.

This is the core of memory optimisation in Pandas. Real datasets often carry columns stored as `float64` or `object` when `int8`, `int16`, `float32`, or `category` would do; a numeric column of IDs might shrink from 8 bytes to 2, and a low-cardinality string column becomes a `category` with dramatic savings. The tools are `to_numeric(downcast=...)`, `astype`, and `astype('category')`, complemented by chunked reads and selective `usecols` at load time. The pattern taught here — measure, downcast, measure again — is precisely how practitioners fit otherwise unmanageable frames into memory, and it completes the survey of reading, ranking, grouping, joining, pivoting, windowing, and reshaping that this challenge collection set out to build.

In [ ]:
df = pd.DataFrame({'v': range(1000)})
before = df.memory_usage(deep=True).sum()
df['v'] = pd.to_numeric(df['v'], downcast='unsigned')
after = df.memory_usage(deep=True).sum()
assert after < before
print(before, '->', after)

## Summary

These ten challenges consolidated the everyday Pandas toolkit, each verified by an assertion so correctness was never in doubt. You inspected a frame with `shape`, `dtypes`, and `isna().sum()`; took the top rows with `nlargest`; combined two reductions in one `groupby().agg(n=..., total=...)`; repaired nulls per group with `groupby().transform(lambda s: s.fillna(s.mean()))`; audited a join using `merge(..., indicator=True)` and the `_merge` column; built a pivoted cross-tab with `margins=True`; smoothed a series with `rolling(3).mean()`; ranked values inside groups with `groupby().rank()`; reshaped wide to long with `melt`; and trimmed memory with `to_numeric(downcast=...)` measured via `memory_usage(deep=True)`. Two meta-skills ran through all ten: know the exact value a call should produce *before* running it, and keep the frame's shape and dtype in mind after every transformation — because those are where most real-world Pandas bugs hide.